In [1]:
import sys
import pathlib
import os
from skmap.catalog import DataCatalog
from skmap.overlay import SpaceOverlay, SpaceTimeOverlay
from skmap.misc import find_files, GoogleSheet, ttprint
from osgeo.gdal import BuildVRT, SetConfigOption
import random
import pandas as pd
import time
import skmap_bindings as sb
import numpy as np
from shapely.geometry import Point
from geopandas import gpd 

folder_path = '/mnt/wodan/global_soc/scikit-map'

base_path = [f'http://192.168.1.{gaia_id}:8333' for gaia_id in range(30,47)]
GDAL_OPTS = {'GDAL_HTTP_VERSION': '1.0', 'CPL_VSIL_CURL_ALLOWED_EXTENSIONS': '.tif'}
max_ram_mb = 1000000
n_threads = 96

# read in gsheet

gsheet_key = '/mnt/apollo/stac/gaia-319808-913d36b5fca4.json'
gsheet_url = 'https://docs.google.com/spreadsheets/d/1lNTpzdHBG5dirYj46iBDRJMk_YAV0Um2ovBc8v3dR9w/edit?gid=78425683#gid=78425683'
gsheet = GoogleSheet(gsheet_key, gsheet_url, verbose=False)

In [2]:
df = pd.read_csv(f'{folder_path}/neospectra_points_v20250406.csv', encoding='latin1', on_bad_lines='skip')
years = df['observation.year'].unique().tolist()
years = [int(ii) for ii in years]

start_months = ['0101', '0201', '0301', '0401', '0501', '0601', '0701', '0801', '0901', '1001', '1101', '1201']
end_moaths = ['0131', '0228', '0331', '0430', '0531', '0630', '0731', '0831', '0930', '1031', '1130', '1231']
# create catalog
catalog = DataCatalog.create_catalog(catalog_def=gsheet.fapar_revisited, years=years, base_path=base_path)

json_out_path = 'test_pot_fapar.json'
catalog.save_json(json_out_path)

In [ ]:
from shapely.geometry import Point
print('data size before overlay', df.shape)
geometry = [Point(xy) for xy in zip(df['longitude.point_wgs84_dd'], df['latitude.point_wgs84_dd'])]
df = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

start = time.time()
space_time_overlay = SpaceTimeOverlay(
        col_date='observation.year',
        points=df, 
        catalog=catalog,
        raster_tiles='ard2_final_status.gpkg',
        verbose=True,
        n_threads=n_threads,
        tile_id_col='TILE')

print(f"Extraction of overlay meta-data: {(time.time() - start):.2f} s")

In [ ]:
start = time.time()
ovelayed_props_data = space_time_overlay.run(gdal_opts=GDAL_OPTS, max_ram_mb=max_ram_mb, out_file_name=f'{folder_path}/ovelayed_landsat_soil.pq')
print(f"Reading overlayed layers: {(time.time() - start):.2f} s")